# 05 — Data Profiling (exercício)

**Objetivo:** praticar consultas exploratórias orientadas a perguntas de negócio.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (exploração sobre o **silver**):

> Fonte → Bronze → Staging → Dim/Fact → Checks → **Profiling** → Marts/Views → Dashboard

Aqui você usa `fact_report` e as dimensões para responder perguntas exploratórias: SLA, distribuição por tipo de ativo, taxa de bounty, qualidade dos writeups. Isso ajuda a desenhar bem os marts no notebook 06 — só dá pra agregar com confiança o que primeiro foi entendido bruto.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete as queries de perfilamento.

In [ ]:
con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")

In [ ]:
perfil_temporal = con.sql("""
SELECT
    COUNT(*) AS reports_total,
    MIN(created_at) AS min_created_at,
    MAX(created_at) AS max_created_at,
    MIN(disclosed_at) AS min_disclosed_at,
    MAX(disclosed_at) AS max_disclosed_at
FROM fact_report
""").df()

sla = con.sql("""
WITH d AS (
    SELECT DATE_DIFF('day', created_at, disclosed_at) AS days_to_disclosure
    FROM fact_report
    WHERE created_at IS NOT NULL
      AND disclosed_at IS NOT NULL
      AND disclosed_at >= created_at
)
SELECT
    AVG(days_to_disclosure) AS avg_days_to_disclosure,
    quantile_cont(days_to_disclosure, 0.50) AS p50_days_to_disclosure,
    quantile_cont(days_to_disclosure, 0.90) AS p90_days_to_disclosure
FROM d
""").df()

asset_type = con.sql("""
SELECT
    COALESCE(a.asset_type, 'sem_asset_type') AS asset_type,
    COUNT(*) AS reports_total,
    COUNT(DISTINCT f.asset_id) AS distinct_assets
FROM fact_report f
LEFT JOIN dim_structured_scope a USING (asset_id)
GROUP BY 1
ORDER BY reports_total DESC
""").df()

bounty_rate = con.sql("""
SELECT
    COUNT(*) AS reports_total,
    SUM(CASE WHEN has_bounty THEN 1 ELSE 0 END) AS bounty_reports,
    100.0 * SUM(CASE WHEN has_bounty THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0) AS bounty_rate_pct
FROM fact_report
""").df()

text_quality = con.sql("""
SELECT
    COUNT(*) AS reports_total,
    SUM(CASE WHEN vulnerability_information IS NULL OR LENGTH(TRIM(vulnerability_information)) = 0 THEN 1 ELSE 0 END) AS missing_writeups,
    SUM(CASE WHEN LENGTH(COALESCE(TRIM(vulnerability_information), '')) BETWEEN 1 AND 199 THEN 1 ELSE 0 END) AS short_writeups,
    AVG(LENGTH(COALESCE(vulnerability_information, ''))) AS avg_writeup_chars
FROM fact_report
""").df()

perfil_temporal, sla, asset_type, bounty_rate, text_quality
